# Cell phenotyping with ImmuVis

This tutorial converts multiplex images and cell-instance masks into per-cell ImmuVis embeddings, then demonstrates a simple cell-type classifier. It includes both methods used in the ImmuVis cell-typing workflow:

- **Patch embeddings:** encode one `32 x 32` crop centered on each cell.
- **Context embeddings:** encode overlapping regions from the complete image and aggregate local tokens by their overlap with each cell mask.

The loading layer is intentionally format-agnostic. You will adapt your files or data objects to the documented NumPy array contract; every transformation after that boundary is explicit and reproducible.

## 0. Installation

From the ImmuVis repository, install the project and a Jupyter kernel:

```bash
uv sync --python 3.12 --extra notebooks
uv run python -m ipykernel install --user --name immuvis --display-name "Python (ImmuVis)"
```

Select the **Python (ImmuVis)** kernel before running the notebook.

The setup cell downloads the model and its matching marker tokenizer from Hugging Face. Choose `MODEL_REPO_ID` from `SzczurekLab/ImmuVis_Conv`, `SzczurekLab/ImmuVis_Swin`, or `SzczurekLab/ImmuVis_ViT`; all three original variants share the same 265-marker mapping. Set `MODEL_REVISION` to a commit hash to pin a release, or leave it as `None` for the default branch. No local checkpoint or tokenizer file is needed.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder, StandardScaler

from multiplex_model.modules import MultiplexAutoencoder

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

MODEL_REPO_ID = "SzczurekLab/ImmuVis_Conv"
MODEL_REVISION = None
MODEL = MultiplexAutoencoder.from_pretrained(
    MODEL_REPO_ID,
    revision=MODEL_REVISION,
    map_location="cpu",
).to(DEVICE)
if MODEL.marker_tokenizer is None:
    raise ValueError("The pretrained model did not provide a marker tokenizer.")
TOKENIZER = MODEL.marker_tokenizer

print(f"Device: {DEVICE}")
print(f"Tokenizer vocabulary: {len(TOKENIZER)} markers")

## 1. Input data contract

ImmuVis does not require a particular source file format in this tutorial. Your loader only needs to produce one or more samples with these values:

- `image`: a numeric NumPy array with shape `(channels, height, width)`. If your source uses another order, set `channel_axis` when adapting it.
- `mask`: a two-dimensional integer array with shape `(height, width)`. Background must be `0`; every positive integer identifies one cell.
- `channel_names`: marker names in exactly the same order as the image channels. Names must match the downloaded `TOKENIZER` mapping exactly.
- `cell_types` (optional): a mapping such as `{cell_id: "cell type"}` for supervised classification.

The adapter below performs the format-dependent boundary work once: it moves the channel axis, validates spatial alignment, removes unsupported channels, preserves the remaining source-channel order, and creates the matching tokenizer IDs. The model receives each channel beside its ID, so sorting channels by token value is neither required nor desired.

Before calling the adapter, handle source-specific details such as TIFF series selection, table column names, axis conventions, or conversion from polygon annotations to an instance mask.

In [ ]:
def prepare_sample(
    image: np.ndarray,
    mask: np.ndarray,
    channel_names: list[str],
    cell_types: dict[int, str] | None = None,
    channel_axis: int = 0,
    tokenizer: dict[str, int] = TOKENIZER,
) -> dict:
    """Convert one user-loaded sample to the representation used below."""
    image = np.asarray(image)
    mask = np.asarray(mask)

    if image.ndim != 3:
        raise ValueError(f"image must have three dimensions; received {image.shape}")
    image = np.moveaxis(image, channel_axis, 0).astype(np.float32, copy=False)

    if mask.ndim != 2:
        raise ValueError(f"mask must have two dimensions; received {mask.shape}")
    if image.shape[1:] != mask.shape:
        raise ValueError(
            f"image and mask spatial shapes differ: {image.shape[1:]} != {mask.shape}"
        )
    if len(channel_names) != image.shape[0]:
        raise ValueError(
            f"received {len(channel_names)} names for {image.shape[0]} image channels"
        )

    mask = mask.astype(np.int32, copy=False)
    if np.any(mask < 0):
        raise ValueError("mask IDs must be zero or positive integers")

    kept_indices = [
        index
        for index, name in enumerate(channel_names)
        if name in tokenizer
    ]
    if not kept_indices:
        raise ValueError("none of the channel names occur in the tokenizer")

    marker_names = [channel_names[index] for index in kept_indices]
    marker_ids = torch.tensor(
        [tokenizer[name] for name in marker_names], dtype=torch.long
    )

    known_cell_ids = set(np.unique(mask)) - {0}
    normalized_cell_types = {
        int(cell_id): str(label)
        for cell_id, label in (cell_types or {}).items()
        if int(cell_id) in known_cell_ids
    }

    return {
        "image": image[kept_indices],
        "mask": mask,
        "marker_names": marker_names,
        "marker_ids": marker_ids,
        "cell_types": normalized_cell_types,
    }

### Connect your loader

Load each image, mask, ordered marker list, and optional annotations with the tools appropriate to your source. Then call `prepare_sample` and place the results in `SAMPLES`.

```python
SAMPLES = [
    prepare_sample(
        image=your_image_array,
        mask=your_instance_mask,
        channel_names=your_ordered_marker_names,
        cell_types=your_cell_id_to_type_mapping,  # optional
        channel_axis=0,
    )
]
```

For several images, append one prepared dictionary per image. Cell IDs only need to be unique within each image. Samples are encoded independently and carry their own marker-token IDs, so their supported marker panels may differ.

In [ ]:
SCHEME = "both"  # "patch", "context", or "both"
CLIP_UPPER_BOUND = 5.0  # set to 99th percentile of the intensity distribution for your dataset (over all pixels)
CROP_SIZE = 32
BATCH_SIZE = 32

# Replace this empty list with prepared samples from your own loader.
SAMPLES = []

## 2. Patch extraction and preprocessing

Patch embeddings use a zero-padded `32 x 32` crop centered at the mean row and column of each cell's mask pixels. Preprocessing is applied after channel alignment and independently to every crop:

1. `arcsinh(x / 5)` compresses the raw intensity range.
2. A low-pass Butterworth filter suppresses high-frequency noise channel by channel.
3. Values are clipped to `[0, CLIP_UPPER_BOUND]` and scaled to `[0, 1]`.

Choose `CLIP_UPPER_BOUND` at dataset level, for example from a high percentile (99th) of the pixel intensities, and use the same value for all samples that are compared.

In [ ]:
def extract_cell_patches(
    img: np.ndarray,
    mask: np.ndarray,
    crop_size: int = 32,
) -> tuple[list[np.ndarray], list[int], np.ndarray]:
    """
    Extract raw cell-centered patches from an image.

    Args:
        img: raw image (C, H, W) — channels already selected/reordered
        mask: segmentation mask (H, W), 0 = background
        crop_size: spatial size of each patch

    Returns:
        patches: list of (C, crop_size, crop_size) arrays (raw, not preprocessed)
        cell_ids: list of cell IDs (sorted)
        centroids: (N, 2) array of (row, col) centroids
    """
    cell_ids = sorted(set(mask.flatten()) - {0})
    half = crop_size // 2
    H, W = mask.shape

    patches = []
    valid_ids = []
    centroids = []

    for cid in cell_ids:
        ys, xs = np.where(mask == cid)
        if len(ys) == 0:
            continue
        cy, cx = int(ys.mean()), int(xs.mean())

        # Compute crop bounds (zero-padded at edges)
        r0 = max(0, cy - half)
        r1 = min(H, cy + half)
        c0 = max(0, cx - half)
        c1 = min(W, cx + half)

        patch = np.zeros((img.shape[0], crop_size, crop_size), dtype=np.float32)
        dr = r1 - r0
        dc = c1 - c0
        pr0 = half - (cy - r0)
        pc0 = half - (cx - c0)
        patch[:, pr0:pr0 + dr, pc0:pc0 + dc] = img[:, r0:r1, c0:c1]

        patches.append(patch)
        valid_ids.append(cid)
        centroids.append((cy, cx))

    print(f"Extracted {len(patches)} raw patches of shape ({img.shape[0]}, {crop_size}, {crop_size})")
    return patches, valid_ids, np.array(centroids)

In [ ]:
def preprocess(img: np.ndarray, clip_upper_bound: float = 5.0) -> np.ndarray:
    """
    Preprocess a raw multiplex image/patch (C, H, W) for ImmuVis models.
    Returns float32 array with values in [0, 1].
    """
    from skimage.filters import butterworth

    x = img.copy()
    x = np.arcsinh(x / 5.0)

    filtered = []
    for ch in range(x.shape[0]):
        filtered.append(butterworth(x[ch], cutoff_frequency_ratio=0.2, high_pass=False))
    x = np.stack(filtered)

    x = np.clip(x, 0, clip_upper_bound) / clip_upper_bound
    return x.astype(np.float32)

## 3. Inspect the loaded ImmuVis model

The setup cell has already loaded the selected Hugging Face checkpoint and its tokenizer. The saved architecture is reconstructed automatically, and the model is in evaluation mode on `DEVICE`. To switch between Conv, Swin, and ViT, change `MODEL_REPO_ID` in the setup cell and rerun the notebook from that cell so the model and prepared samples use the matching tokenizer.

In [ ]:
print(f"Loaded {MODEL_REPO_ID}; latent dimension: {MODEL.latent_dim}")

## 4. Patch embeddings

Each raw cell crop is preprocessed independently, stacked into batches, and paired with the retained marker-token IDs. `model.encode` returns a spatial latent map. Averaging its two spatial dimensions gives one embedding vector per cell, in the same order as the returned cell IDs.

In [ ]:
def run_patch(
    model: MultiplexAutoencoder,
    raw_patches: list[np.ndarray],
    cell_ids: list[int],
    marker_ids: torch.Tensor,
    device: torch.device,
    clip_upper_bound: float = 5.0,
    batch_size: int = 32,
) -> np.ndarray:
    """
    Encode cell patches and return per-cell embeddings.

    Pipeline per patch: preprocess → encode → average spatial dims.
    """
    # Preprocess each raw patch
    preprocessed = [preprocess(p, clip_upper_bound) for p in raw_patches]

    all_embs = []
    for i in range(0, len(preprocessed), batch_size):
        batch = preprocessed[i:i + batch_size]
        x = torch.from_numpy(np.stack(batch)).to(device)  # (B, C, 32, 32)
        ch = marker_ids.unsqueeze(0).expand(len(batch), -1).to(device)

        with torch.no_grad():
            out = model.encode(x, ch)["output"]  # (B, E, H', W')
            emb = out.mean(dim=(2, 3))  # (B, E)

        all_embs.append(emb.cpu().numpy())

    embeddings = np.concatenate(all_embs, axis=0)
    print(f"Patch embeddings: {embeddings.shape}")
    return embeddings

## 5. Context embeddings

Context inference preprocesses the complete channel-aligned image once, pads both image and mask, and divides the image into overlapping `128 x 128` crops. Each crop is split into `8 x 8` image patches before encoding.

The encoded patch tokens are L2-normalized and assigned to every cell whose mask overlaps that patch. The final embedding for a cell is the area-weighted mean of all assigned tokens across overlapping crops. This preserves local surroundings while returning one vector per segmented cell.

In [ ]:
def _get_uniform_crops(
    img: np.ndarray, stride: int, crop_size: int = 128
) -> tuple[list, list]:
    """Extract overlapping crops from an image (C, H, W)."""
    _, H, W = img.shape
    crops, indices = [], []
    for r in range(0, H - crop_size + 1, stride):
        for c in range(0, W - crop_size + 1, stride):
            crops.append(img[:, r:r + crop_size, c:c + crop_size])
            indices.append((r, c))
    return crops, indices


def _assign_to_cells(
    patch_tokens: np.ndarray, crop_mask: np.ndarray, patch_size: int
) -> tuple[dict, dict]:
    """Assign patch tokens to cells based on mask overlap."""
    grid_h, grid_w, dim = patch_tokens.shape
    cell_tokens, cell_weights = {}, {}

    for gi in range(grid_h):
        for gj in range(grid_w):
            r0, c0 = gi * patch_size, gj * patch_size
            patch_mask = crop_mask[r0:r0 + patch_size, c0:c0 + patch_size]
            unique, counts = np.unique(patch_mask, return_counts=True)
            for uid, cnt in zip(unique, counts):
                if uid == 0:
                    continue
                cell_tokens.setdefault(int(uid), []).append(patch_tokens[gi, gj])
                cell_weights.setdefault(int(uid), []).append(float(cnt))

    return cell_tokens, cell_weights


def run_context(
    model: MultiplexAutoencoder,
    raw_img: np.ndarray,
    mask: np.ndarray,
    marker_ids: torch.Tensor,
    device: torch.device,
    clip_upper_bound: float = 5.0,
    crop_size: int = 128,
    patch_size: int = 8,
    stride: int = 42,
) -> tuple[np.ndarray, list[int]]:
    """
    Run context-based inference on a full image.

    Pipeline: preprocess full image → pad → crop → encode patches → assign to cells
    """
    # Preprocess the full image (channel reorder already done)
    img = preprocess(raw_img, clip_upper_bound)

    H, W = mask.shape

    # Pad image and mask (matching celltyping: pad_size=120 for context)
    pad = 120
    img_pad = np.pad(img, ((0, 0), (pad, pad), (pad, pad)), mode="constant")
    mask_pad = np.pad(mask, (pad, pad), mode="constant")

    # Extract overlapping crops
    crops, crop_indices = _get_uniform_crops(img_pad, stride, crop_size)

    cell_tokens_all, cell_weights_all = {}, {}

    for i, crop_np in enumerate(crops):
        # Divide crop into patches
        patches = []
        for pr in range(0, crop_size, patch_size):
            for pc in range(0, crop_size, patch_size):
                patches.append(crop_np[:, pr:pr + patch_size, pc:pc + patch_size])

        x = torch.from_numpy(np.stack(patches)).to(device)
        ch = marker_ids.unsqueeze(0).expand(len(patches), -1).to(device)

        with torch.no_grad():
            out = model.encode(x, ch)["output"]
            patch_tokens = out.mean(dim=(2, 3))
            patch_tokens = torch.nn.functional.normalize(patch_tokens, p=2, dim=1)
            patch_tokens = patch_tokens.cpu().numpy()

        # Reshape to grid
        grid_h = crop_size // patch_size
        grid_w = crop_size // patch_size
        patch_tokens = patch_tokens.reshape(grid_h, grid_w, -1)

        # Get crop mask
        r0, c0 = crop_indices[i]
        crop_mask = mask_pad[r0:r0 + crop_size, c0:c0 + crop_size]

        # Assign patches to cells
        ct, cw = _assign_to_cells(patch_tokens, crop_mask, patch_size)
        for cid, toks in ct.items():
            cell_tokens_all.setdefault(cid, []).extend(toks)
            cell_weights_all.setdefault(cid, []).extend(cw[cid])

    # Area-weighted averaging per cell
    cell_ids = sorted(cell_tokens_all.keys())
    embeddings = []
    for cid in cell_ids:
        toks = np.stack(cell_tokens_all[cid])
        weights = np.array(cell_weights_all[cid])
        weights = weights / weights.sum()
        emb = (toks * weights[:, None]).sum(axis=0)
        embeddings.append(emb)

    embeddings = np.stack(embeddings)
    print(f"Context embeddings: {embeddings.shape}")
    return embeddings, cell_ids

## 6. Extract embeddings

The loop below applies both methods to every prepared sample. Results remain grouped by sample so that local cell IDs can be matched safely to annotations. The final arrays concatenate embeddings across samples for classification and visualization.

In [ ]:
patch_results = []
context_results = []

for sample_index, sample in enumerate(SAMPLES):
    print(f"\nSample {sample_index + 1}/{len(SAMPLES)}")
    image = sample["image"]
    mask = sample["mask"]
    marker_ids = sample["marker_ids"]

    raw_patches, cell_ids, _ = extract_cell_patches(
        image, mask, crop_size=CROP_SIZE
    )

    if SCHEME in ("patch", "both"):
        embeddings = run_patch(
            MODEL,
            raw_patches,
            cell_ids,
            marker_ids,
            DEVICE,
            clip_upper_bound=CLIP_UPPER_BOUND,
            batch_size=BATCH_SIZE,
        )
        patch_results.append(
            {
                "embeddings": embeddings,
                "cell_ids": cell_ids,
                "cell_types": sample["cell_types"],
            }
        )

    if SCHEME in ("context", "both"):
        embeddings, context_cell_ids = run_context(
            MODEL,
            image,
            mask,
            marker_ids,
            DEVICE,
            clip_upper_bound=CLIP_UPPER_BOUND,
        )
        context_results.append(
            {
                "embeddings": embeddings,
                "cell_ids": context_cell_ids,
                "cell_types": sample["cell_types"],
            }
        )


def concatenate_results(results: list[dict]) -> np.ndarray:
    if not results:
        return np.empty((0, MODEL.latent_dim), dtype=np.float32)
    return np.concatenate([result["embeddings"] for result in results], axis=0)


all_patch_embs = concatenate_results(patch_results)
all_ctx_embs = concatenate_results(context_results)

if not SAMPLES:
    print("SAMPLES is empty. Add data through prepare_sample before extraction.")
else:
    print(f"Total patch embeddings: {all_patch_embs.shape}")
    print(f"Total context embeddings: {all_ctx_embs.shape}")

## 7. Cell-type classification

Annotations are joined to embeddings within each sample by the positive integer in the instance mask. Unannotated cells remain available in the embedding output but are omitted from classification.

The example standardizes embedding dimensions and fits a class-balanced logistic regression on all labeled embeddings. The reported scores are training-set diagnostics, not estimates of generalization. For a scientific evaluation, split independent samples before fitting the scaler and classifier.

In [ ]:
def collect_labeled_embeddings(
    results: list[dict],
) -> tuple[np.ndarray, np.ndarray]:
    """Align optional cell-type annotations with per-cell embeddings."""
    embeddings = []
    labels = []

    for result in results:
        for index, cell_id in enumerate(result["cell_ids"]):
            if cell_id in result["cell_types"]:
                embeddings.append(result["embeddings"][index])
                labels.append(result["cell_types"][cell_id])

    if not embeddings:
        return np.empty((0, MODEL.latent_dim), dtype=np.float32), np.array([])
    return np.stack(embeddings), np.asarray(labels)


def run_classification(
    embeddings: np.ndarray,
    labels: np.ndarray,
    scheme_name: str,
):
    """Run simple logistic regression and print metrics."""
    if len(labels) == 0:
        print(f"\n  [{scheme_name}] No matching labels found — skipping classification")
        return

    X_all = embeddings
    y_all = labels

    # Filter NaNs
    nan_mask = np.isnan(X_all).any(axis=1)
    if nan_mask.any():
        print(f"  Removing {nan_mask.sum()} NaN samples")
        X_all = X_all[~nan_mask]
        y_all = y_all[~nan_mask]

    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y_all)
    class_names = le.classes_

    # Scale + classify (same as celltyping: StandardScaler + LogisticRegression)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_all)

    clf = LogisticRegression(max_iter=5000, class_weight="balanced", random_state=42)
    clf.fit(X_scaled, y_encoded)
    y_pred = clf.predict(X_scaled)

    # Metrics
    acc = accuracy_score(y_encoded, y_pred)
    bal_acc = balanced_accuracy_score(y_encoded, y_pred)
    macro_f1 = f1_score(y_encoded, y_pred, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_encoded, y_pred, average="weighted", zero_division=0)

    print(f"\n  [{scheme_name}] Classification (fit on all data):")
    print(f"    Samples: {len(y_all)} | Classes: {len(class_names)}")
    print(f"    Accuracy:          {acc:.4f}")
    print(f"    Balanced accuracy: {bal_acc:.4f}")
    print(f"    Macro F1:          {macro_f1:.4f}")
    print(f"    Weighted F1:       {weighted_f1:.4f}")

    # Per-class F1
    f1_per_class = f1_score(y_encoded, y_pred, average=None, zero_division=0)
    print(f"    Per-class F1:")
    for name, f1 in zip(class_names, f1_per_class):
        print(f"      {name:20s} {f1:.4f}")


patch_X, patch_y = collect_labeled_embeddings(patch_results)
context_X, context_y = collect_labeled_embeddings(context_results)

if len(patch_y) > 0:
    run_classification(patch_X, patch_y, "patch")
if len(context_y) > 0:
    run_classification(context_X, context_y, "context")
if len(patch_y) == 0 and len(context_y) == 0:
    print("No cell-type annotations were supplied; classification was skipped.")

## 8. Visualize embeddings

A two-dimensional t-SNE view is useful for a qualitative check of embedding structure. At most 2,000 cells per method are plotted to keep the example responsive.

In [ ]:
plot_data = {}
if len(all_patch_embs) > 0:
    plot_data["patch"] = all_patch_embs
if len(all_ctx_embs) > 0:
    plot_data["context"] = all_ctx_embs

n = len(plot_data)
if n > 0:
    fig, axes = plt.subplots(1, n, figsize=(6 * n, 5))
    if n == 1:
        axes = [axes]

    for ax, (scheme, emb) in zip(axes, plot_data.items()):
        if emb.shape[0] < 2:
            ax.text(0.5, 0.5, "Need >1 cell", ha="center", va="center")
            ax.set_title(scheme)
            continue

        # Subsample for speed if too many cells
        max_points = 2000
        if emb.shape[0] > max_points:
            idx = np.random.RandomState(42).choice(emb.shape[0], max_points, replace=False)
            emb_plot = emb[idx]
        else:
            emb_plot = emb

        perp = min(30, emb_plot.shape[0] - 1)
        emb_2d = TSNE(n_components=2, perplexity=perp, random_state=42).fit_transform(emb_plot)
        ax.scatter(emb_2d[:, 0], emb_2d[:, 1], s=5, alpha=0.3)
        ax.set_title(f"{scheme} — {emb.shape[0]} cells, dim={emb.shape[1]}")
        ax.set_xlabel("t-SNE 1")
        ax.set_ylabel("t-SNE 2")

    plt.tight_layout()
    plt.savefig("tutorial_celltyping_embeddings.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("Saved: tutorial_celltyping_embeddings.png")
else:
    print("No embeddings are available to visualize.")

In [ ]:
print("\n--- Summary ---")
print(f"Model: {MODEL_REPO_ID}")
print(f"Samples prepared: {len(SAMPLES)}")
if SAMPLES:
    panel_sizes = [len(sample["marker_names"]) for sample in SAMPLES]
    print(f"Retained marker channels per sample: {panel_sizes}")
if len(all_patch_embs) > 0:
    print(f"Patch: {all_patch_embs.shape[0]} cells, dim={all_patch_embs.shape[1]}")
if len(all_ctx_embs) > 0:
    print(f"Context: {all_ctx_embs.shape[0]} cells, dim={all_ctx_embs.shape[1]}")